In [14]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

df = pd.read_csv('car_fuel_efficiency_2026.csv')

In [15]:
df = df.fillna(0)

In [16]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values
y_test = df_test.fuel_efficiency_mpg.values

del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

In [17]:
train_dicts = df_train.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

feature_names = dv.get_feature_names_out()

In [18]:
dt = DecisionTreeRegressor(max_depth=1, random_state=1)
dt.fit(X_train, y_train)

root_feature_idx = dt.tree_.feature[0]
raw_feature_name = feature_names[root_feature_idx]
orig_feature_name = raw_feature_name.split('=')[0]

print(f"{orig_feature_name} (raw feature: {raw_feature_name})\n")

model_year (raw feature: model_year)



In [19]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))

print(f"{rmse:.4f}")

1.8332


In [20]:
for n in [10, 50, 100, 150]:
    rf_n = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf_n.fit(X_train, y_train)
    y_pred_n = rf_n.predict(X_val)
    rmse_n = np.sqrt(mean_squared_error(y_val, y_pred_n))
    print(f"n_estimators={n:3d}: {rmse_n:.3f}")

n_estimators= 10: 1.833
n_estimators= 50: 1.774
n_estimators=100: 1.770
n_estimators=150: 1.769


In [ ]:
for depth in [10, 15, 20, 25]:
    rmses = []
    for n in [10, 50, 100, 150]:
        rf_d = RandomForestRegressor(n_estimators=n, max_depth=depth, random_state=1, n_jobs=-1)
        rf_d.fit(X_train, y_train)
        y_pred_d = rf_d.predict(X_val)
        rmses.append(np.sqrt(mean_squared_error(y_val, y_pred_d)))
    mean_rmse = np.mean(rmses)
    print(f"max_depth={depth:2d}: {mean_rmse:.3f}")

max_depth=10: 1.754
max_depth=15: 1.784
max_depth=20: 1.787


In [ ]:
rf = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

target_features = ['vehicle_weight', 'horsepower', 'acceleration', 'engine_displacement']
importances_dict = {}

for name, imp in zip(feature_names, rf.feature_importances_):
    base_col = name.split('=')[0]
    importances_dict[base_col] = importances_dict.get(base_col, 0.0) + imp

for feat in target_features:
    print(f"{feat:19s}: {importances_dict.get(feat, 0.0):.4f}")

vehicle_weight     : 0.2033
horsepower         : 0.0783
acceleration       : 0.0550
engine_displacement: 0.0613


In [ ]:
clean_feature_names = [f.replace('[', '_').replace(']', '_').replace('<', '_') for f in feature_names]
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=clean_feature_names)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=clean_feature_names)

watchlist = [(dtrain, 'train'), (dval, 'val')]

for eta in [0.3, 0.1]:
    xgb_params = {
        'eta': eta,
        'max_depth': 6,
        'min_child_weight': 1,
        'objective': 'reg:squarederror',
        'nthread': 8,
        'seed': 1,
        'verbosity': 0
    }
    model = xgb.train(xgb_params, dtrain, num_boost_round=100, evals=watchlist, verbose_eval=False)
    y_pred_xgb = model.predict(dval)
    rmse_xgb = np.sqrt(mean_squared_error(y_val, y_pred_xgb))
    print(f"eta={eta:.1f}: {rmse_xgb:.4f}")

eta=0.3: 1.8265
eta=0.1: 1.7248
